<a href="https://colab.research.google.com/github/samarae2508-design/week4_communityproject/blob/main/week4_assignment_communityproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics paho-mqtt

In [2]:
YOUR_NAME = "samra987"     # <- CHANGE THIS (example: Sathwik042)

BROKER = "broker.hivemq.com"  # backup brokers: broker.emqx.io, test.mosquitto.org

import re
assert (
    re.fullmatch(r"[a-z0-9]+", YOUR_NAME) and YOUR_NAME != "yourname123"
), "Change YOUR_NAME to your own first name + roll number (lowercase letters and numbers only)"

TOPIC = f"ssse/w4/{YOUR_NAME}/cmd"
print("Your topic is:")
print(TOPIC)
print("\nCopy this EXACT text into the TOPIC line of the Wokwi code.")

Your topic is:
ssse/w4/samra987/cmd

Copy this EXACT text into the TOPIC line of the Wokwi code.


In [ ]:
#include <WiFi.h>
#include <PubSubClient.h>
#include <time.h>

// WiFi details
const char* WIFI_SSID = "Wokwi-GUEST";
const char* WIFI_PASS = "";

// MQTT details - FIXED URL HERE
const char* BROKER = "://hivemq.com";
const int MQTT_PORT = 1883;

WiFiClient espClient;
PubSubClient client(espClient);

// Buzzer pin
const int BUZZER_PIN = 3;


// Bell schedule (24-hour format)
struct BellSchedule {
  int hour;
  int minute;
  const char* event;
};

BellSchedule schedule[] = {
  {10, 30, "Morning Break"},
  {10, 45, "Break End"},
  {13, 0, "Lunch Start"},
  {13, 40, "Lunch End"},
  {16, 0, "College End"}
};

const int totalEvents = sizeof(schedule) / sizeof(schedule[0]);

int lastTriggeredDay[5] = {-1, -1, -1, -1, -1};

// FIXED WIFI CONNECTION FUNCTION
void connectWiFi() {
  WiFi.begin(WIFI_SSID, WIFI_PASS);
  Serial.print("Connecting to WiFi");

  while (WiFi.status() != WL_CONNECTED) {
    delay(500);
    Serial.print(".");
  }

  Serial.println("");
  Serial.println("WiFi Connected");
}

void connectMQTT() {
  while (!client.connected()) {
    if (client.connect("ESP32CollegeBell")) {
      Serial.println("MQTT Connected");
    } else {
      delay(2000);
    }
  }
}

// FIXED AUDIO FREQUENCY AND TOPIC FUNCTION
void ringBell(const char* event) {
  Serial.print("Bell ringing: ");
  Serial.println(event);

  if (client.connected()) {
    client.publish("ssse/w4/samra987/cmd", event);
  }

  // --- 2-BEEP BELL SEQUENCE ---
  tone(BUZZER_PIN, 1000); // Beep 1 (1 kHz frequency)
  delay(1000);
  noTone(BUZZER_PIN);
  delay(500);             // Break

  tone(BUZZER_PIN, 1000); // Beep 2
  delay(1000);
  noTone(BUZZER_PIN);
}

void setup() {
  Serial.begin(115200);

  pinMode(BUZZER_PIN, OUTPUT);
  digitalWrite(BUZZER_PIN, LOW);

  // --- MOVE THIS TO THE TOP TO TEST SOUND IMMEDIATELY ---
  ringBell("Testing Sound System");

  connectWiFi();

  client.setServer(BROKER, MQTT_PORT);

  configTime(19800, 0, "pool.ntp.org", "://google.com");

  Serial.println("Automatic College Bell System Started");
}

void loop() {
  if (!client.connected()) {
    connectMQTT();
  }

  client.loop();

  struct tm timeInfo;

  if (!getLocalTime(&timeInfo, 1000)) {
    Serial.println("Waiting for time...");
    return;
  }

  for (int i = 0; i < totalEvents; i++) {
    if (timeInfo.tm_hour == schedule[i].hour &&
        timeInfo.tm_min == schedule[i].minute &&
        lastTriggeredDay[i] != timeInfo.tm_yday) {

      lastTriggeredDay[i] = timeInfo.tm_yday;
      ringBell(schedule[i].event);
    }
  }

  delay(1000);
}
